# 06 - BGE-M3 Embeddings

## Objective

Generate embeddings for Pipeline 2 using:

- Section-Based Chunking
- BAAI/bge-m3

Pipeline:

Documents
→ Section-Based Chunking
→ BGE-M3 Embeddings
→ FAISS
→ Qwen3:14B

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

## Load Oracle ERP Documents

Load source documents and validate page counts.

In [2]:
from src.config import DOCS_PATH
from src.loaders.document_loader import DocumentLoader

loader = DocumentLoader(DOCS_PATH)

documents = loader.load()

print(
    f"Source Pages: {len(documents):,}"
)

Starting Document Loading...

Loading PDF : iSupplier_portal.pdf

Loading PDF : Purchasing_manual.pdf


DOCUMENT LOADING SUMMARY
PDF Files Loaded   : 2
DOCX Files Loaded  : 0
XLSX Files Loaded  : 0
Total Documents    : 1502
Source Pages: 1,502


## Create Section-Based Chunks

Create larger context-preserving chunks aligned with Oracle ERP business processes.

In [3]:
from langchain_core.documents import Document

In [4]:
def create_section_chunks(
    documents,
    target_size=2000
):

    section_chunks = []

    current_text = ""

    current_metadata = None

    for doc in documents:

        page_text = doc.page_content.strip()

        if not page_text:
            continue

        if current_metadata is None:
            current_metadata = doc.metadata.copy()

        current_text += "\n\n" + page_text

        if len(current_text) >= target_size:

            section_chunks.append(
                Document(
                    page_content=current_text.strip(),
                    metadata=current_metadata
                )
            )

            current_text = ""
            current_metadata = None

    if current_text:

        section_chunks.append(
            Document(
                page_content=current_text.strip(),
                metadata=current_metadata
            )
        )

    return section_chunks

In [5]:
section_chunks = create_section_chunks(
    documents
)

print(
    f"Section Chunks Created: {len(section_chunks):,}"
)

Section Chunks Created: 880


## Verify Available Hardware

Automatically detect CPU or GPU execution.

In [6]:
import torch

In [7]:
print("=" * 80)

print("DEVICE INFORMATION")

print("=" * 80)

print(
    f"CUDA Available : {torch.cuda.is_available()}"
)

if torch.cuda.is_available():

    print(
        f"GPU : {torch.cuda.get_device_name(0)}"
    )

DEVICE INFORMATION
CUDA Available : True
GPU : NVIDIA RTX A5000 Laptop GPU


## Determine Embedding Device

In [8]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    f"Embedding Device: {device}"
)

Embedding Device: cuda


## Load BGE-M3 Embedding Model

Pipeline 2 uses BAAI/bge-m3, a larger multilingual embedding model designed for advanced retrieval scenarios.

In [9]:
from langchain_huggingface import (
    HuggingFaceEmbeddings
)

In [10]:
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={
        "device": device
    }
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

## Generate Validation Embedding

In [11]:
sample_embedding = embedding_model.embed_query(
    "What is Oracle Purchasing?"
)

In [12]:
print("=" * 80)

print("EMBEDDING VALIDATION")

print("=" * 80)

print(
    f"Embedding Dimension: {len(sample_embedding)}"
)

EMBEDDING VALIDATION
Embedding Dimension: 1024


In [13]:
print(
    sample_embedding[:10]
)

[-0.09190966933965683, 0.0033375495113432407, -0.03528357297182083, 0.05672512575984001, -0.021620074287056923, 0.012939447537064552, 0.0033292402513325214, -0.01666775345802307, 0.02717924863100052, 0.020425498485565186]


## Validate Chunk Embeddings

In [14]:
sample_chunk = section_chunks[0]

In [15]:
sample_vector = embedding_model.embed_query(
    sample_chunk.page_content
)

In [16]:
print("=" * 80)

print("CHUNK EMBEDDING CHECK")

print("=" * 80)

print(
    f"Vector Length: {len(sample_vector)}"
)

CHUNK EMBEDDING CHECK
Vector Length: 1024


In [17]:
print("=" * 80)

print("CHUNK PREVIEW")

print("=" * 80)

print(
    sample_chunk.page_content[:1000]
)

CHUNK PREVIEW
Oracle® iSupplier Portal
 User's Guide 
Release 12.2
 Part No. E48972-13
October 2025

Oracle iSupplier Portal User's Guide , Release 12.2
Part No. E48972-13
Copyright © 2009, 2025, Oracle and/or its affiliates. 
Primary Author:     Chetna Arora
Contributing Author:     Gowri Arur, Pragya Nair, Smile Nagpal
This software and related documentation are provided under a license agreement containing restrictions on 
use and disclosure and are protected by intellectual property laws. Except as expressly permitted in your 
license agreement or allowed by law, you may not use, copy, reproduce, translate, broadcast, modify, license, 
transmit, distribute, exhibit, perform, publish, or display any part, in any form, or by any means. Reverse 
engineering, disassembly, or decompilation of this software, unless required by law for interoperability, is 
prohibited. 
The information contained herein is subject to change without notice and is not warranted to be error-free. If 
you find

## Export Embedding Metadata

In [18]:
embedding_info = {
    "Model": "BAAI/bge-m3",
    "Dimensions": len(sample_embedding),
    "Chunk Count": len(section_chunks),
    "Device": device
}

In [19]:
import pandas as pd

embedding_df = pd.DataFrame(
    embedding_info.items(),
    columns=[
        "Property",
        "Value"
    ]
)

embedding_df

,Property,Value
0,Model,BAAI/bge-m3
1,Dimensions,1024
2,Chunk Count,880
3,Device,cuda


In [20]:
from pathlib import Path

output_dir = Path(
    "../screenshots/embeddings"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

output_file = (
    output_dir
    / "bgem3_embedding_info.csv"
)

embedding_df.to_csv(
    output_file,
    index=False
)

print(
    f"Saved: {output_file.resolve()}"
)

Saved: D:\AI\Projects\oracle-rag-comparison-lab\screenshots\embeddings\bgem3_embedding_info.csv


# Conclusion

## BGE-M3 Embeddings

Model:

- BAAI/bge-m3

Characteristics:

- High-capacity embeddings
- 1024-dimensional vectors
- Multilingual
- Designed for advanced retrieval scenarios

## Pipeline

Section-Based Chunking
→ BGE-M3 Embeddings
→ FAISS
→ Qwen3:14B

## Next Step

Proceed to:

### 07_Chroma_Indexing.ipynb